In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
PROJECT_ROOT = Path('.')
DATA_DIR = PROJECT_ROOT / 'data'
ISTAT_DIR = DATA_DIR / 'istat'


def clean_unnamed(df: pd.DataFrame) -> pd.DataFrame:
    return df.loc[:, ~df.columns.astype(str).str.startswith('Unnamed')].copy()


def load_csv(path: Path, date_cols: list[str] | None = None) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df = clean_unnamed(df)
    for c in (date_cols or []):
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors='coerce')
    return df


def overview(df: pd.DataFrame, name: str) -> pd.DataFrame:
    mem_mb = df.memory_usage(deep=True).sum() / 1024**2
    return pd.DataFrame([{'table': name, 'rows': len(df), 'cols': df.shape[1], 'memory_mb': round(mem_mb, 2)}])


def nulls(df: pd.DataFrame, name: str) -> pd.DataFrame:
    out = pd.DataFrame(
        {'column': df.columns, 'null_pct': (df.isna().mean() * 100).values})
    out['table'] = name
    return out.sort_values(by='null_pct', ascending=False)

In [ ]:
tables = {
    'utenti': load_csv(DATA_DIR / 'utenti_reduced.csv', ['DtaRegUserData_datetime', 'DtaNascitaUser', 'DtaPresuntoParto']),
    'rewusers': load_csv(DATA_DIR / 'rewusers_reduced.csv', ['created_at', 'updated_at', 'lastActivity', 'appFirstAccess', 'siteLastDate', 'appLastDate', 'created_at_2021']),
    'accessi': load_csv(DATA_DIR / 'accessi_reduced.csv', ['created_at', 'updated_at']),
    'codici': load_csv(DATA_DIR / 'codici_reduced.csv', ['date_created_at', 'created_at', 'updated_at']),
    'missioni': load_csv(DATA_DIR / 'missioni_reduced.csv', ['created_at', 'updated_at', 'deadline', 'startDate']),
    'premi': load_csv(DATA_DIR / 'premi_reduced.csv', ['datarichiestapremio', 'created_at_premio']),
    'prodotti': load_csv(DATA_DIR / 'Anagrafica_prodotti_digital.csv'),
}

feature_dict = pd.read_excel(DATA_DIR / 'Features_Dictionary.xlsx')

print('Loaded tables:')
for k, v in tables.items():
    print(f'  - {k}: {v.shape[0]:,} rows, {v.shape[1]} cols')

feature_dict.head(10)

In [ ]:
inventory = pd.concat([overview(df, name)
                      for name, df in tables.items()], ignore_index=True)
inventory.sort_values(by='rows', ascending=False)

In [ ]:
null_report = pd.concat([nulls(df, name)
                        for name, df in tables.items()], ignore_index=True)

dup_rows = pd.DataFrame([
    {'table': name, 'duplicate_rows': int(df.duplicated().sum())}
    for name, df in tables.items()
])

key_uniqueness = pd.DataFrame([
    {'table': 'utenti', 'key': 'idSSO',
        'is_unique': tables['utenti']['idSSO'].is_unique},
    {'table': 'rewusers', 'key': 'idSSO',
        'is_unique': tables['rewusers']['idSSO'].is_unique},
    {'table': 'rewusers', 'key': 'userId',
        'is_unique': tables['rewusers']['userId'].is_unique},
])

missing_top = (
    null_report.groupby(['table', 'column'], as_index=False)
    .agg(null_pct=('null_pct', 'max'))
    .sort_values(by=['table', 'null_pct'], ascending=[True, False])
)

fig_missing = px.bar(
    missing_top.query('null_pct > 0'),
    x='column',
    y='null_pct',
    color='table',
    title='Missingness by Column (null % > 0)',
)
fig_missing.update_layout(xaxis_tickangle=-55, height=520)

dup_rows, key_uniqueness, missing_top.head(30), fig_missing

In [ ]:
rew = tables['rewusers'][['userId', 'idSSO']].dropna()
map_user_to_idsso = rew.drop_duplicates(
    subset='userId').set_index('userId')['idSSO']

one_to_one_report = pd.DataFrame([
    {'check': 'rewusers userId unique',
        'value': bool(rew['userId'].is_unique)},
    {'check': 'rewusers idSSO unique', 'value': bool(rew['idSSO'].is_unique)},
    {'check': 'rows in rewusers bridge', 'value': int(len(rew))},
])

codici_user = tables['codici']['userId']
premi_user = tables['premi']['userid']

codici_mapped = codici_user.map(map_user_to_idsso)
premi_mapped = premi_user.map(map_user_to_idsso)

join_quality_report = pd.DataFrame([
    {'table': 'codici', 'rows': int(len(codici_user)), 'mapped_rows': int(
        codici_mapped.notna().sum()), 'unmapped_rows': int(codici_mapped.isna().sum())},
    {'table': 'premi', 'rows': int(len(premi_user)), 'mapped_rows': int(
        premi_mapped.notna().sum()), 'unmapped_rows': int(premi_mapped.isna().sum())},
])
join_quality_report['mapped_pct'] = (
    join_quality_report['mapped_rows'] / join_quality_report['rows'] * 100).round(2)

one_to_one_report, join_quality_report

In [ ]:
accessi = tables['accessi'].copy()
codici = tables['codici'].copy()
missioni = tables['missioni'].copy()
premi = tables['premi'].copy()

accessi['event_date'] = pd.to_datetime(accessi['created_at'], errors='coerce')
codici['event_date'] = pd.to_datetime(codici['created_at'], errors='coerce')
missioni['event_date'] = pd.to_datetime(
    missioni['created_at'], errors='coerce')
premi['event_date'] = pd.to_datetime(
    premi['datarichiestapremio'], errors='coerce')

temporal_horizon_report = pd.DataFrame([
    {'table': 'accessi', 'min_date': accessi['event_date'].min(
    ), 'max_date': accessi['event_date'].max()},
    {'table': 'codici', 'min_date': codici['event_date'].min(
    ), 'max_date': codici['event_date'].max()},
    {'table': 'missioni', 'min_date': missioni['event_date'].min(
    ), 'max_date': missioni['event_date'].max()},
    {'table': 'premi', 'min_date': premi['event_date'].min(
    ), 'max_date': premi['event_date'].max()},
])

monthly = []
for name, df in [('accessi', accessi), ('codici', codici), ('missioni', missioni), ('premi', premi)]:
    s = (
        df.dropna(subset=['event_date'])
          .assign(month=lambda x: x['event_date'].dt.to_period('M').dt.to_timestamp())
          .groupby('month', as_index=False)
          .size()
          .rename(columns={'size': 'events'})
    )
    s['table'] = name
    monthly.append(s)
monthly_events = pd.concat(monthly, ignore_index=True)

fig_monthly = px.line(monthly_events, x='month', y='events',
                      color='table', markers=True, title='Monthly Event Volumes')
fig_monthly.update_layout(height=480)

temporal_horizon_report, fig_monthly

In [ ]:
utenti = tables['utenti']
rewusers = tables['rewusers']

sanity = pd.DataFrame([
    {'metric': 'totalPoints null %', 'value': round(
        rewusers['totalPoints'].isna().mean() * 100, 2)},
    {'metric': 'lastActivity null %', 'value': round(
        rewusers['lastActivity'].isna().mean() * 100, 2)},
    {'metric': 'ETA_MM_BambinoTODAY < 0 count', 'value': int(
        (utenti['ETA_MM_BambinoTODAY'] < 0).sum())},
])

fig_points = px.histogram(rewusers, x='totalPoints',
                          nbins=60, title='totalPoints Distribution')
fig_points.update_layout(height=420)

eta = utenti[['ETA_MM_BambinoTODAY']].copy()
eta['lifecycle_bucket'] = pd.cut(
    eta['ETA_MM_BambinoTODAY'],
    bins=[-999, -0.0001, 11, 23, 29, 35, 999],
    labels=['pregnancy', '0-11', '12-23', '24-29', '30-35', '36+']
)
fig_eta = px.histogram(eta, x='ETA_MM_BambinoTODAY', nbins=45,
                       color='lifecycle_bucket', title='Child Age Distribution')
fig_eta.update_layout(height=420)

region_counts = utenti['Regione'].fillna(
    'UNKNOWN').value_counts().reset_index()
region_counts.columns = ['Regione', 'users']
fig_region = px.bar(region_counts.head(25), x='Regione',
                    y='users', title='Top Regions by User Count')
fig_region.update_layout(xaxis_tickangle=-45, height=480)

sanity, fig_points, fig_eta, fig_region

In [ ]:
active_users = utenti[['idSSO']].drop_duplicates().copy()

access_users = set(tables['accessi']['idsso'].dropna().astype(str))
mission_users = set(tables['missioni']['idsso'].dropna().astype(str))
scan_users = set(codici_mapped.dropna().astype(str))
redeem_users = set(premi_mapped.dropna().astype(str))

active_users['idSSO_str'] = active_users['idSSO'].astype(str)
active_users['has_access'] = active_users['idSSO_str'].isin(access_users)
active_users['has_scan'] = active_users['idSSO_str'].isin(scan_users)
active_users['has_mission'] = active_users['idSSO_str'].isin(mission_users)
active_users['has_redeem'] = active_users['idSSO_str'].isin(redeem_users)

coverage = pd.DataFrame([
    {'behavior': 'access', 'pct_users': round(
        active_users['has_access'].mean() * 100, 2)},
    {'behavior': 'scan', 'pct_users': round(
        active_users['has_scan'].mean() * 100, 2)},
    {'behavior': 'mission', 'pct_users': round(
        active_users['has_mission'].mean() * 100, 2)},
    {'behavior': 'redeem', 'pct_users': round(
        active_users['has_redeem'].mean() * 100, 2)},
])

stack = pd.DataFrame([
    {'behavior': 'access', 'status': 'has_event',
        'users': int(active_users['has_access'].sum())},
    {'behavior': 'access', 'status': 'no_event',
        'users': int((~active_users['has_access']).sum())},
    {'behavior': 'scan', 'status': 'has_event',
        'users': int(active_users['has_scan'].sum())},
    {'behavior': 'scan', 'status': 'no_event',
        'users': int((~active_users['has_scan']).sum())},
    {'behavior': 'mission', 'status': 'has_event',
        'users': int(active_users['has_mission'].sum())},
    {'behavior': 'mission', 'status': 'no_event',
        'users': int((~active_users['has_mission']).sum())},
    {'behavior': 'redeem', 'status': 'has_event',
        'users': int(active_users['has_redeem'].sum())},
    {'behavior': 'redeem', 'status': 'no_event',
        'users': int((~active_users['has_redeem']).sum())},
])

fig_cov = px.bar(stack, x='behavior', y='users', color='status',
                 barmode='stack', title='Behavioral Coverage (Users)')
fig_cov.update_layout(height=420)

coverage, fig_cov

In [ ]:
req_cols = ['FREQ', 'REF_AREA', 'Territorio',
            'DATA_TYPE', 'TIME_PERIOD', 'Osservazione']

income_path = ISTAT_DIR / \
    'Regioni e tipo di comune (IT1,32_292_DF_DCCV_REDNETFAMFONTERED_9,1.0).csv'
family_path = ISTAT_DIR / \
    'Tipologie familiari - regioni e tipo comune (IT1,82_87_DF_DCCV_AVQ_FAMIGLIE_12,1.0).csv'

income = pd.read_csv(income_path, encoding='utf-8-sig', usecols=lambda c: c in (
    set(req_cols + ['IMPUTED_RENTS', 'FAM_MAIN_INCOME_SOURCE'])))
family = pd.read_csv(family_path, encoding='utf-8-sig',
                     usecols=lambda c: c in (set(req_cols + ['MEASURE'])))

income = income[(income['DATA_TYPE'] == 'REDD_MEDIANO_FAM') & (income['IMPUTED_RENTS'].astype(
    str) == '1') & (income['FAM_MAIN_INCOME_SOURCE'].astype(str) == '9')].copy()
income['TIME_PERIOD'] = pd.to_numeric(income['TIME_PERIOD'], errors='coerce')
income_latest_year = int(income['TIME_PERIOD'].max())
income_latest = income[income['TIME_PERIOD'] == income_latest_year].copy()

family = family[(family['DATA_TYPE'] == 'AVTYPE_HYESCHI')
                & (family['MEASURE'] == 'HSC_N')].copy()
family['TIME_PERIOD'] = pd.to_numeric(family['TIME_PERIOD'], errors='coerce')
family_latest_year = int(family['TIME_PERIOD'].max())
family_latest = family[family['TIME_PERIOD'] == family_latest_year].copy()


def normalize(s): return (
    s.astype(str)
     .str.upper()
     .str.replace("'", '', regex=False)
     .str.replace('-', ' ', regex=False)
     .str.replace('/', ' ', regex=False)
     .str.replace(r'\s+', ' ', regex=True)
     .str.strip()
)


user_regions = utenti[['idSSO', 'Regione']].copy()
user_regions['region_key'] = normalize(user_regions['Regione'].fillna(''))

income_latest['region_key'] = normalize(income_latest['Territorio'])
family_latest['region_key'] = normalize(family_latest['Territorio'])

region_map = {'VALLE DAOSTA': 'VALLE DAOSTA',
              'TRENTINO ALTO ADIGE': 'TRENTINO ALTO ADIGE'}
user_regions['region_key'] = user_regions['region_key'].replace(region_map)
income_latest['region_key'] = income_latest['region_key'].replace(region_map)
family_latest['region_key'] = family_latest['region_key'].replace(region_map)

income_small = income_latest[['region_key', 'Territorio', 'TIME_PERIOD', 'Osservazione']].rename(
    columns={'Osservazione': 'regional_median_income_eur'})
family_small = family_latest[['region_key', 'Territorio', 'TIME_PERIOD', 'Osservazione']].rename(
    columns={'Osservazione': 'share_families_with_children_pct'})

istat_enrichment_preview = (
    user_regions[['idSSO', 'Regione', 'region_key']]
    .merge(income_small[['region_key', 'regional_median_income_eur']], on='region_key', how='left')
    .merge(family_small[['region_key', 'share_families_with_children_pct']], on='region_key', how='left')
)

region_match_stats = pd.DataFrame([
    {'feature': 'regional_median_income_eur', 'match_pct': round(
        istat_enrichment_preview['regional_median_income_eur'].notna().mean() * 100, 2)},
    {'feature': 'share_families_with_children_pct', 'match_pct': round(
        istat_enrichment_preview['share_families_with_children_pct'].notna().mean() * 100, 2)},
])

income_small.head(10), family_small.head(10), region_match_stats